# Solar Prediction Colab Full Run

Run the cleaned package against the full local solar/weather dataset from Google Drive. This is a sanity full run, not a final modeling benchmark.

Expected data path inside the uploaded Drive project: `data/solar_weather.csv`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Update this if your Drive folder name differs.
PROJECT_DIR = '/content/drive/MyDrive/solar_prediction'
DATA_PATH = 'data/solar_weather.csv'
OUTPUT_DIR = 'artifacts/colab_full_run_actual_data'

# First serious-ish run defaults. Adjust upward after this completes cleanly.
EPOCHS = 20
HIDDEN_DIM = 32
BATCH_SIZE = 256
HORIZON_STEPS = 1  # 1=15min, 4=1h, 16=4h, 96=24h for 15-minute data.
SEASONAL_LAG = 96  # Daily cycle for 15-minute data.


In [ ]:
import os
from pathlib import Path

project = Path(PROJECT_DIR)
assert project.exists(), f'Project folder not found: {project}'
os.chdir(project)

data_path = Path(DATA_PATH)
assert data_path.exists(), f'Full dataset not found: {data_path}'

print('Working directory:', Path.cwd())
print('Full dataset:', data_path)


Use a fresh Colab runtime. This notebook installs the local package only and leaves Colab's scientific Python stack alone.

In [ ]:
%pip install -q --no-deps -e .


In [ ]:
import pandas as pd

preview = pd.read_csv(data_path, nrows=5)
time_col = pd.read_csv(data_path, usecols=['Time'])['Time']
parsed_time = pd.to_datetime(time_col, errors='coerce')

print('Rows:', len(time_col))
print('Columns:', preview.columns.tolist())
print('Time range:', parsed_time.min(), 'to', parsed_time.max())
preview.head()


In [ ]:
import shlex
import subprocess
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
cmd = [
    'solar-predict',
    'compare',
    '--data', str(data_path),
    '--epochs', str(EPOCHS),
    '--hidden-dim', str(HIDDEN_DIM),
    '--batch-size', str(BATCH_SIZE),
    '--horizon-steps', str(HORIZON_STEPS),
    '--seasonal-lag', str(SEASONAL_LAG),
    '--device', device,
    '--output', OUTPUT_DIR,
    '--quiet',
]

print('Device:', device)
print('Command:', shlex.join(cmd))
result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError(f'Command failed with exit code {result.returncode}')


In [ ]:
from io import StringIO

output_lines = result.stdout.splitlines()
header_idx = next(i for i, line in enumerate(output_lines) if line.startswith('model,'))
metrics = pd.read_csv(StringIO('\n'.join(output_lines[header_idx:])))
metrics = metrics.sort_values('rmse').reset_index(drop=True)
metrics


In [ ]:
import json
import matplotlib.pyplot as plt

output_dir = Path(OUTPUT_DIR)
metrics_json = output_dir / 'comparison_metrics.json'
assert metrics_json.exists(), f'Metrics JSON was not written: {metrics_json}'

metrics.to_csv(output_dir / 'comparison_metrics_table.csv', index=False)
with metrics_json.open() as f:
    saved_metrics = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
metrics.plot.bar(x='model', y='rmse', ax=axes[0], legend=False, color='#4C78A8')
axes[0].set_title('RMSE')
axes[0].set_xlabel('')
axes[0].set_ylabel('W/m^2')

metrics.plot.bar(x='model', y='r2', ax=axes[1], legend=False, color='#59A14F')
axes[1].set_title('R²')
axes[1].set_xlabel('')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print('Saved metrics:', metrics_json)
print('Best RMSE model:', metrics.iloc[0]['model'])
